In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP = "/content/drive/MyDrive/reuse_vq_finetune.zip"
assert os.path.exists(ZIP), f"Upload reuse_vq_finetune.zip to Drive root first ({ZIP} missing)"
!unzip -qo "$ZIP" -d /content/
%cd /content/reuse_vq_finetune
!ls -la
!chmod +x urgent2025_challenge/prepare_espnet_data.sh urgent2025_challenge/utils/*.sh urgent2025_challenge/utils/*.pl 2>/dev/null; true
!cd urgent2025_challenge && git --no-pager diff --stat || echo "(.git not bundled; see data_prep/PREPARE.md)"


In [ ]:
import os
%cd /content/reuse_vq_finetune/urgent2025_challenge

!apt-get -qq update && apt-get -qq install -y ffmpeg sox flac zip > /dev/null
!pip install -q espnet soundfile soxr librosa scipy tqdm pyloudnorm onnxruntime

!bash prepare_espnet_data.sh

print("\n=== Data generation finished ===")
!echo "train pairs:" && wc -l simulation_train/log/meta.tsv
!echo "val pairs:"   && wc -l simulation_validation/log/meta.tsv
!du -sh simulation_train simulation_validation 2>/dev/null


In [ ]:
import os
%cd /content/reuse_vq_finetune/urgent2025_challenge
os.makedirs("/content/drive/MyDrive/reuse_data", exist_ok=True)
BACKUP = "/content/drive/MyDrive/reuse_data/simulation_data.tar"
!tar cf "$BACKUP" \
    simulation_train/log/meta.tsv simulation_train/clean simulation_train/noisy \
    simulation_validation/log/meta.tsv simulation_validation/clean simulation_validation/noisy
print("Backed up to:", BACKUP)
!ls -lh "$BACKUP"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP = "/content/drive/MyDrive/reuse_vq_finetune.zip"
!unzip -qo "$ZIP" -d /content/
%cd /content/reuse_vq_finetune/urgent2025_challenge

BACKUP = "/content/drive/MyDrive/reuse_data/simulation_data.tar"
if not os.path.exists("simulation_train/log/meta.tsv"):
    assert os.path.exists(BACKUP), f"No data on disk and no backup at {BACKUP}"
    !tar xf "$BACKUP"
!echo "train pairs:" && wc -l simulation_train/log/meta.tsv
!echo "val pairs:"   && wc -l simulation_validation/log/meta.tsv


In [ ]:
import torch, sys, re, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

!pip install -q huggingface_hub safetensors soundfile pyyaml tqdm librosa soxr tensorboard pesq pystoi joblib
!pip install -q transformers==4.40.0
!pip install -q loralib   # required by vox-profile's WhisperWrapper (VQ metric)

torch_ver = re.match(r'(\d+\.\d+)', torch.__version__).group(1)
py_ver = f"cp{''.join(str(x) for x in sys.version_info[:2])}"
cxx_abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
print(f"Detected: torch{torch_ver}, {py_ver}, cxx11abi{cxx_abi}")

def try_install_wheel(urls, fallback_pip):
    for url in urls:
        print(f"  Trying: {url.split('/')[-1]}")
        r = subprocess.run([sys.executable, "-m", "pip", "install", url], capture_output=True, text=True)
        if r.returncode == 0:
            print("  OK!"); return True
        print("  Failed")
    print("  Building from source...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + fallback_pip)
    return True

causal_urls = [f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1%2Bcu12torch{torch_ver}cxx11abi{cxx_abi}-{py_ver}-{py_ver}-linux_x86_64.whl"]
mamba_urls = [f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1%2Bcu12torch{torch_ver}cxx11abi{cxx_abi}-{py_ver}-{py_ver}-linux_x86_64.whl"]
print("\nInstalling causal-conv1d:"); try_install_wheel(causal_urls, ["causal-conv1d>=1.4.0", "--no-build-isolation"])
print("\nInstalling mamba-ssm:");     try_install_wheel(mamba_urls,  ["mamba-ssm", "--no-build-isolation"])

print("\n" + "="*60)
print("  ALL INSTALLED.  NOW: Runtime -> Restart runtime, then run Cell 6 onward.")
print("="*60)


In [ ]:
import torch, transformers, mamba_ssm
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}")
print(f"vRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"transformers {transformers.__version__} | mamba-ssm {mamba_ssm.__version__}")
assert transformers.__version__.startswith("4.40"), "Wrong transformers version"
print("Environment OK.")


In [ ]:
import os
%cd /content/reuse_vq_finetune/finetune
from huggingface_hub import snapshot_download

snapshot_download(repo_id="nvidia/RE-USE", local_dir="./REUSE", local_dir_use_symlinks=False)
print("RE-USE downloaded.")

if not os.path.exists('./vox-profile-release/src'):
    !git clone https://github.com/tiantiaf0627/vox-profile-release.git ./vox-profile-release
snapshot_download(repo_id="tiantiaf/whisper-large-v3-voice-quality",
                  local_dir="./vox-profile-release", local_dir_use_symlinks=False)
print("Voice-quality model downloaded.")


In [ ]:
import yaml, os
%cd /content/reuse_vq_finetune/finetune

FULL_META = "/content/reuse_vq_finetune/urgent2025_challenge/simulation_train/log/meta.tsv"
META = "/content/reuse_vq_finetune/urgent2025_challenge/simulation_train/log/meta_train_10k.tsv"
assert os.path.exists(FULL_META), f"full meta.tsv missing: {FULL_META} -- re-run Cell 4 restore (train audio not restored)."
if not os.path.exists(META):
    !python make_subset.py --src "$FULL_META" --dst "$META" --n 10000 --seed 42
assert os.path.exists(META), f"subset still missing: {META}"

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

VAL_META = "/content/reuse_vq_finetune/urgent2025_challenge/simulation_validation/log/meta.tsv"
cfg['data']['train_meta_tsv'] = META
cfg['data']['data_root'] = ""            # auto -> .../urgent2025_challenge
cfg['data']['test_meta_tsv'] = VAL_META  # dev-clean validation (disjoint speakers)
cfg['seed'] = 42
cfg['training']['epochs'] = 1
cfg['training']['batch_size'] = 4        # safe on A100-80GB with Whisper VQ in memory
cfg['training']['num_workers'] = 4
cfg['training']['validate_every_n_steps'] = 500
cfg['training']['early_stop_patience'] = 100000   # disabled -> arms run identical steps
cfg['training']['checkpoint_dir'] = "/content/drive/MyDrive/reuse_data/checkpoints"
cfg['training']['log_dir'] = "/content/drive/MyDrive/reuse_data/logs"

with open("config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("Config updated. train_meta_tsv =", META)


!python train.py --config config.yaml --arm C2 --resume auto
!python train.py --config config.yaml --arm C1 --resume auto


In [ ]:
%cd /content/reuse_vq_finetune/finetune
!pip install -q scipy

CKPT_DIR = "/content/drive/MyDrive/reuse_data/checkpoints"

!python evaluate.py --config config.yaml \
    --ckpt C2={CKPT_DIR}/C2/epoch_1.pt \
    --ckpt C1={CKPT_DIR}/C1/epoch_1.pt \
    --ckpt B0={CKPT_DIR}/B0_draft1/epoch_1.pt \
    --ckpt B1={CKPT_DIR}/B1_W0.2/epoch_1.pt \
    --out-dir /content/drive/MyDrive/reuse_data/eval_iter3 \
    --examples 5


In [ ]:
%cd /content/reuse_vq_finetune/finetune
!pip install -q onnxruntime pesq scipy

!cd /content/reuse_vq_finetune/urgent2025_challenge && bash utils/download_dnsmos_onnx.sh

CKPT_DIR = "/content/drive/MyDrive/reuse_data/checkpoints"
!python evaluate_independent.py --config config.yaml \
    --ckpt C2={CKPT_DIR}/C2/epoch_1.pt \
    --ckpt C1={CKPT_DIR}/C1/epoch_1.pt \
    --ckpt B0={CKPT_DIR}/B0_draft1/epoch_1.pt \
    --ckpt B1={CKPT_DIR}/B1_W0.2/epoch_1.pt \
    --dnsmos-dir /content/reuse_vq_finetune/urgent2025_challenge/DNSMOS/DNSMOS \
    --out-dir /content/drive/MyDrive/reuse_data/eval_independent


In [ ]:
%cd /content/reuse_vq_finetune/finetune
!pip install -q gdown scipy jiwer "torchmetrics[audio]" openai-whisper speechbrain requests

import os, zipfile, tarfile, shutil, gdown
FILE_ID = "1e8H8QKaCyztbYRv7GBMkAlFBceHkvnrp"
OOD_DIR = "/content/daic_ood"
ARCHIVE = "/content/daic_ood_archive"

if os.path.isdir(OOD_DIR) and any(f.endswith(('.wav', '.flac')) for _, _, fs in os.walk(OOD_DIR) for f in fs):
    print(f"[ood] {OOD_DIR} already populated -- skipping download")
else:
    shutil.rmtree(OOD_DIR, ignore_errors=True); os.makedirs(OOD_DIR, exist_ok=True)
    gdown.download(id=FILE_ID, output=ARCHIVE, quiet=False)
    if zipfile.is_zipfile(ARCHIVE):
        with zipfile.ZipFile(ARCHIVE) as z: z.extractall(OOD_DIR)
    elif tarfile.is_tarfile(ARCHIVE):
        with tarfile.open(ARCHIVE) as t: t.extractall(OOD_DIR)
    else:                                   # a single audio file, not an archive
        shutil.move(ARCHIVE, os.path.join(OOD_DIR, "clip.wav"))

n_wav = sum(1 for _, _, fs in os.walk(OOD_DIR) for f in fs if f.lower().endswith(('.wav', '.flac')))
print(f"[ood] audio files under {OOD_DIR}: {n_wav}")

CKPT_DIR = "/content/drive/MyDrive/reuse_data/checkpoints"
!python evaluate_ood.py --config config.yaml \
    --ood-dir /content/daic_ood \
    --ckpt C2={CKPT_DIR}/C2/epoch_1.pt \
    --ckpt C1={CKPT_DIR}/C1/epoch_1.pt \
    --whisper-model base \
    --clip-seconds 15 \
    --out-dir /content/drive/MyDrive/reuse_data/eval_ood
